# Day 069 · 成长因子
**Growth Factors** · 阶段 P3 · 数据与因子工程

> 上一节我们认识了价值因子,挑「便宜」的票;这一节换个完全相反的思路,挑「长得快」的票,这就是成长因子。成长因子盯的是一家公司赚钱的速度,营收今年比去年涨了多少、利润这几年是不是越滚越大。一家店去年卖一百万、今年卖一百三十万,这就是营收同比涨了30%;要是连着好几年都在往上冲,这家公司就处在扩张期,股价往往跟着水涨船高。这一节我们用十只真实股票,五只是公认的高成长公司,光伏、锂电、疫苗、制冷部件、模拟芯片;另外五只是稳但不长的低成长公司,铁路、水电、港口、高速、贸易。我们会算出每只票多年的复合增速,把高低成长分成两组拉回测,看高成长组是不是真的涨得更猛;同时也亲眼看一个反直觉的点,成长股虽然涨得快,但更颠、回撤更深,估值对利率和市场情绪特别敏感,这正是追高成长股最容易吃的亏。

---

**课件生成日期:** 2026-06-26  ·  **建议学习时长:** 22 分钟

学习路径建议:1)先看视频建立直觉 → 2)阅读本 notebook 跑代码 → 3)看 PDF 课件复习要点 → 4)做自测题

## 🔧 第一步:环境自检 + 自动安装

**第一次拿到这份 notebook,请先运行下面这一格。** 它会:
1. 检查所有需要的 Python 包,缺什么自动 `pip install` 装上
2. 注入中文字体到 matplotlib(让图标不出乱码)
3. 跑完看到 `✓ 环境就绪` 就可以继续下面的代码

> 这一格只在第一次跑要等几十秒,后面再开 notebook 就秒过。

In [ ]:
# === 环境自检 + 自动安装(运行此单元格即可) ===
# 检测缺失的库 → 自动 pip 安装 → 注入中文字体 → 一行命令搞定
import importlib
import subprocess
import sys
import os

REQUIRED = ["baostock", "matplotlib", "numpy", "numpy_financial", "pandas", "scipy", "sklearn", "statsmodels", "yfinance"]
PIP_NAME = {
    "sklearn": "scikit-learn",
    "cv2": "opencv-python",
    "PIL": "Pillow",
    "bs4": "beautifulsoup4",
    "yaml": "PyYAML",
}

missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(PIP_NAME.get(mod, mod))

if missing:
    print(f"⏳ 缺少 {len(missing)} 个包,正在自动安装:{missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("✓ 安装完成")
else:
    print(f"✓ 所有 {len(REQUIRED)} 个必需库已就绪")

# === 中文字体配置(让 matplotlib 不出乱码) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

CJK_FONT_PATHS = [
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",  # Linux/WSL
    "C:/Windows/Fonts/msyh.ttc",                               # Windows 微软雅黑
    "C:/Windows/Fonts/simhei.ttf",                             # Windows 黑体
    "/System/Library/Fonts/PingFang.ttc",                      # macOS 苹方
    "/System/Library/Fonts/STHeiti Medium.ttc",                # macOS 黑体
]
for p in CJK_FONT_PATHS:
    if os.path.exists(p):
        fm.fontManager.addfont(p)
        print(f"✓ 中文字体已加载:{os.path.basename(p)}")
        break
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP", "Microsoft YaHei",
                                    "PingFang SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("✓ 环境就绪 — 现在可以跑下面的代码单元格")


## 学习目标

- 搞懂成长因子到底看什么:营收同比、净利润同比、多年复合增长,衡量的是一家公司赚钱「长得多快」
- 学会算复合年增长率CAGR:把好几年的增长摊成一个平均速度,比只看某一年的暴涨暴跌更靠谱
- 用十只真实股票把高成长和低成长分成两组,亲手做一次月度调仓回测,看成长组涨得是不是真更猛
- 理解一个反直觉的事:成长股涨得猛但更颠,回撤更深,估值对利率和情绪特别敏感
- 建立纪律:追成长股不能只看涨幅,得同时看波动和回撤,涨得快的票往往也是跌得狠的票

## 历史背景:小李追高成长票,牛市里赚得眉开眼笑,一波情绪退潮直接腰斩,才懂成长股的颠

小李是个新股民,听人说成长股涨得快,就专挑营收利润增速最高的票买。一开始那阵子市场情绪好,他买的几只光伏、锂电票一路往上冲,半年账户涨了一大截,他逢人就说自己找到了赚钱密码,觉得选股就该选长得最快的。可好景不长,后来市场情绪一退潮、利率一抬头,他手里这些高成长票跌得比谁都狠,短短几个月就把前面赚的全吐回去,还倒亏了不少,账户直接腰斩。他百思不得其解，明明公司业绩还在涨啊,怎么股价跌成这样。一位老股民点醒他:成长股贵就贵在大家把未来好几年的好预期都提前算进了股价里,所以平时市盈率高得吓人;一旦情绪变了、利率涨了,这份预期最先被砍,股价自然跌得最猛。涨得快和跌得狠,本来就是一枚硬币的两面。小李这才明白,成长因子不是不能用,是得知道它的脾气,涨得猛、也颠得狠。这节课我们就用真实数据把这个脾气量出来给你看。



## 核心概念

下面每一节是听完视频后回头细读的内容。

### 1. 1. 营收同比:一家公司今年比去年多卖了多少

营收同比,说白了就是拿今年的营业收入跟去年同期比，看涨了百分之多少。举个最小的例子:一家奶茶店去年一整年卖了一百万，今年卖了一百三十万，那营收同比就是用130减100，再除以100，等于涨了30%。这个数字直接告诉你这家公司的生意是在变大还是变小。营收同比越高，说明公司扩张越快、市场越认它的产品。成长因子最基础的1 块，就是看营收同比，挑那些生意越做越大的公司。


### 2. 2. 复合年增长率CAGR:把好几年的增长摊成一个平均速度

只看某一年的增长容易被骗，因为有的公司这一年暴涨、下一年又暴跌。CAGR，也就是复合年增长率，是把好几年的总增长摊成每年平均涨多少，更稳更可信。就好比你存钱，3 年从一万变成两万，不是简单除以三说每年涨三千多，而是问每年按固定百分比滚，滚3 年正好翻倍，那个固定百分比大约是26%，这就是复合增长。算法是用末尾值除以起始值，开几年的方根，再减一。CAGR 平滑掉了单年的暴涨暴跌，更能反映一家公司真实的长期扩张速度。


### 3. 3. 研发投入占比:成长公司舍不舍得为未来花钱

成长因子除了看营收利润涨多快，还常看一个钱花在哪的指标，研发投入占比，也就是公司每年花多少钱搞研发，占营收的比例。简单讲，一家科技公司、芯片公司、医药公司，要想持续长大，就得不停往研发里砸钱，开发新产品、攒新技术。研发投入占比高，说明这家公司舍得为未来下注，成长的底子更厚；要是一家公司营收涨得快但几乎不投研发，这种成长往往不持久。所以看成长，不光看现在长得多快，还要看它有没有为继续长大埋下伏笔。


### 4. 4. 成长股为什么对利率和情绪特别敏感

这是成长因子最该懂的一点。成长股贵，是因为大家把它未来好几年的好日子都提前算进了今天的股价里，所以市盈率往往高得吓人。打个比方，这就像你提前付了一张3 年后才能用的演唱会门票，价格自然贵。可一旦利率涨了，未来的钱折算到今天就更不值钱了；或者市场情绪一变，大家对未来没那么乐观了，这份提前透支的预期最先被砍掉，股价就跌得最猛。所以成长股是涨得快、也跌得狠的典型，利率一动、情绪一变，它比谁都颠。


### 5. 5. 质量和成长常常同向,但不是一回事

新手容易把成长和质量搞混。成长看的是长得多快，营收利润增速；质量看的是赚钱的质地好不好，比如净资产收益率高不高、现金流稳不稳。这俩经常同向，一家又快又好的公司，往往既是高成长又是高质量。但它们不是一回事:有的公司增速很猛，可是靠不停借钱、烧钱换来的，质量其实很差，一旦扩张踩刹车就原形毕露。所以成长因子要和质量因子搭着看，光快不行，还得赚得健康。下一节我们就专门讲质量因子，这里先记住，成长是速度，质量是质地，两个维度。


## 实操:成长因子:用净利润同比PIT打分、多年营收/净利复合增速CAGR分组,高成长 vs 低成长月度调仓回测,再看成长股估值对情绪利率更敏感(波动/回撤更大、PE大起大落)

下面这段代码跟视频里讲解的 highlights 是一致的,可以**直接 Run All** 看结果。

**依赖安装:**
```bash
pip install pandas numpy matplotlib yfinance akshare statsmodels
```


In [ ]:
# day_069_growth_factor.py — 成长因子:营收增速 / 净利润CAGR,高成长 vs 低成长分组 + 估值敏感
# 真名上屏:baostock / query_growth_data → YOYNI(净利润同比) / query_profit_data → netProfit, MBRevenue(营收) / pubDate(披露日) PIT / peTTM / nlargest 分组 / 月末调仓回测
# 核心类比:成长因子像看一家店「翻台率涨多快」;CAGR(复合年增长率)是把好几年的增长摊成一个平均速度;成长股涨得猛但更颠,估值对利率/情绪更敏感
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import baostock as bs


def _data_path(_name):
    # 铁律62:data/ 放在 notebook 文件夹里。仓库根(run_lab)存取 out/notebook/data/;
    # 原版 notebook 在 out/notebook/ 用 data/;中国版在 out/notebook/cn/ 用 ../data/
    from pathlib import Path as _P
    _here = _P.cwd()
    for _b in [_here / 'data', _here / '..' / 'data', _here / 'out' / 'notebook' / 'data', _here / '..' / '..' / 'data', _here / '..' / '..' / '..' / 'data']:
        if (_b / _name).exists():
            return str(_b / _name)
    if (_here / 'out' / 'notebook').exists():
        _t = _here / 'out' / 'notebook' / 'data'
    elif _here.name == 'cn':
        _t = _here / '..' / 'data'
    else:
        _t = _here / 'data'
    _t.mkdir(parents=True, exist_ok=True)
    return str(_t / _name)


pd.set_option('display.width', 160)
plt.rcParams['axes.unicode_minus'] = False

# ==== 标的池:5 只高成长(光伏/锂电/疫苗/制冷部件/模拟芯片)+ 5 只低成长(铁路/水电/港口/高速/贸易)====
HIGH = {'晶澳科技': 'sz.002459', '国轩高科': 'sz.002074', '智飞生物': 'sz.300122', '三花智控': 'sz.002050', '圣邦股份': 'sz.300661'}
LOW = {'大秦铁路': 'sh.601006', '华能水电': 'sh.600025', '上港集团': 'sh.600018', '皖通高速': 'sh.600012', '建发股份': 'sh.600153'}
STOCKS = {**HIGH, **LOW}
START, END = '2018-01-01', '2024-12-31'
CSV_PX = _data_path('D069_growth_close.csv')   # 前复权收盘 + peTTM(算收益 + 估值敏感)
CSV_FIN = _data_path('D069_growth.csv')        # 季度财报:pubDate/statDate/YOYNI(净利同比)/netProfit(净利润)/MBRevenue(营收)


# ==== 0. 拉数据:前复权收盘+peTTM,季度净利同比+净利润+营收及其披露日 ====
def _q(fn, **kw):
    # baostock 偶发瞬时报错(尤其并发/调用多时)→ 最多 3 次,失败就重新登录再试,保证不漏数据
    rs = fn(**kw)
    for _ in range(3):
        if rs.error_code == '0':
            return rs
        try:
            bs.logout()
        except Exception:
            pass
        bs.login()
        rs = fn(**kw)
    return rs


def _fetch():
    bs.login()
    px, pe = {}, {}
    fin_rows = []
    for name, code in STOCKS.items():
        rs = _q(bs.query_history_k_data_plus, code=code, fields='date,close,peTTM', start_date=START, end_date=END, frequency='d', adjustflag='2')
        r = []
        while rs.error_code == '0' and rs.next():
            r.append(rs.get_row_data())
        s = pd.DataFrame(r, columns=['date', 'close', 'peTTM'])
        s['close'] = pd.to_numeric(s['close'], errors='coerce')
        s['peTTM'] = pd.to_numeric(s['peTTM'], errors='coerce')
        px[name] = s.set_index('date')['close']
        pe[name] = s.set_index('date')['peTTM']
        for y in range(2017, 2025):
            for q in range(1, 5):
                # 成长:净利润同比 YOYNI + 披露日 pubDate
                rg = _q(bs.query_growth_data, code=code, year=y, quarter=q)
                yoyni, pub_g, stat_g = '', '', ''
                while rg.error_code == '0' and rg.next():
                    g = rg.get_row_data()   # code,pubDate,statDate,YOYEquity,YOYAsset,YOYNI,YOYEPSBasic,YOYPNI
                    pub_g, stat_g, yoyni = g[1], g[2], g[5]
                # 盈利:净利润 netProfit + 营收 MBRevenue + 披露日(用于 CAGR 与营收同比)
                rp = _q(bs.query_profit_data, code=code, year=y, quarter=q)
                netp, mbrev, pub_p, stat_p = '', '', '', ''
                while rp.error_code == '0' and rp.next():
                    p = rp.get_row_data()   # code,pubDate,statDate,roeAvg,npMargin,gpMargin,netProfit,epsTTM,MBRevenue,totalShare,liqaShare
                    pub_p, stat_p, netp, mbrev = p[1], p[2], p[6], p[8]
                fin_rows.append({
                    'name': name,
                    'pubDate': pub_g or pub_p,
                    'statDate': stat_g or stat_p,
                    'YOYNI': yoyni,
                    'netProfit': netp,
                    'MBRevenue': mbrev,
                })
    bs.logout()
    px_df = pd.DataFrame(px)
    pe_df = pd.DataFrame(pe)
    px_df.columns = ['close__' + c for c in px_df.columns]
    pe_df.columns = ['peTTM__' + c for c in pe_df.columns]
    merged = pd.concat([px_df, pe_df], axis=1)
    return merged, pd.DataFrame(fin_rows)


if os.path.exists(CSV_PX) and os.path.exists(CSV_FIN):
    print('[数据] 从本地 CSV 读取 收盘/peTTM + 季度财报')
    merged = pd.read_csv(CSV_PX, parse_dates=['date']).set_index('date')
    fin = pd.read_csv(CSV_FIN)
else:
    print('[数据] baostock 拉取 10 只票 前复权收盘+peTTM + 季度净利同比/净利润/营收 ...')
    merged, fin = _fetch()
    merged.index.name = 'date'
    merged.to_csv(CSV_PX)
    fin.to_csv(CSV_FIN, index=False)
    print('[数据] 已存 CSV ->', CSV_PX)

merged.index = pd.to_datetime(merged.index)
px = merged[[c for c in merged.columns if c.startswith('close__')]].copy()
pe = merged[[c for c in merged.columns if c.startswith('peTTM__')]].copy()
px.columns = [c.replace('close__', '') for c in px.columns]
pe.columns = [c.replace('peTTM__', '') for c in pe.columns]
px = px[[n for n in STOCKS if n in px.columns]]
pe = pe[[n for n in STOCKS if n in pe.columns]]
pe = pe.where(pe > 0)   # 亏损季 peTTM 为负无意义,剔除,保证估值水平曲线干净可读

fin['pubDate'] = pd.to_datetime(fin['pubDate'], errors='coerce')
fin['statDate'] = pd.to_datetime(fin['statDate'], errors='coerce')
for col in ['YOYNI', 'netProfit', 'MBRevenue']:
    fin[col] = pd.to_numeric(fin[col], errors='coerce')
fin = fin.dropna(subset=['pubDate', 'statDate'])

print('\n==== 成长因子:高成长 vs 低成长分组 + 估值敏感 ====')
print('标的池 %d 只(高成长 %d / 低成长 %d) · %s ~ %s' % (
    len(STOCKS), len(HIGH), len(LOW), px.index[0].date(), px.index[-1].date()))

# ==== 1. 成长信号 PIT:净利润同比 YOYNI 挂到披露日 pubDate,ffill 到月末(每月当时已知的最新增速)====
monthly = px.resample('ME').last()                 # 月末收盘
mret = monthly.pct_change().shift(-1)              # 本月末建仓 -> 下月末收益
pe_m = pe.resample('ME').last()                    # 月末 peTTM


def signal_panel(date_col, value_col):
    panel = {}
    for name in STOCKS:
        sub = fin[fin['name'] == name].dropna(subset=[value_col]).sort_values(date_col)
        if len(sub) == 0:
            panel[name] = pd.Series(np.nan, index=monthly.index)
            continue
        ser = pd.Series(sub[value_col].values, index=sub[date_col])
        ser = ser[~ser.index.duplicated(keep='last')].sort_index()
        panel[name] = ser.reindex(monthly.index, method='ffill')
    return pd.DataFrame(panel)


sig_growth = signal_panel('pubDate', 'YOYNI')      # PIT 成长信号:净利润同比

# ==== 2. 多年 净利润CAGR + 营收CAGR(年度首->末复合年增长率)做静态选股表 ====
fin['year'] = fin['statDate'].dt.year
# 取每只票每年「年报口径」最后一条(年末 statDate 为该年 12 月)做年度值
annual_np = {}
annual_rev = {}
for name in STOCKS:
    sub = fin[fin['name'] == name].dropna(subset=['statDate'])
    # 只用年报口径(报告期末为 12-31 = Q4 全年累计),避免拿到半年/三季报的不完整年份扭曲 CAGR
    annual = sub[sub['statDate'].dt.month == 12].sort_values('statDate')
    np_year = annual.set_index('year')['netProfit'].dropna()
    rev_year = annual.set_index('year')['MBRevenue'].dropna()
    annual_np[name] = np_year[~np_year.index.duplicated(keep='last')]
    annual_rev[name] = rev_year[~rev_year.index.duplicated(keep='last')]


def _cagr(series):
    s = series.dropna()
    if len(s) < 2:
        return np.nan
    start_v, end_v = s.iloc[0], s.iloc[-1]
    years = s.index[-1] - s.index[0]
    if years <= 0 or start_v <= 0 or end_v <= 0:   # 守卫:负利润/缺口不算 CAGR
        return np.nan
    return (end_v / start_v) ** (1.0 / years) - 1.0


cagr_rows = []
for name in STOCKS:
    cagr_np = _cagr(annual_np[name])
    cagr_rev = _cagr(annual_rev[name])
    cagr_rows.append({'name': name, '组别': '高成长' if name in HIGH else '低成长',
                      '净利CAGR%': cagr_np * 100 if pd.notna(cagr_np) else np.nan,
                      '营收CAGR%': cagr_rev * 100 if pd.notna(cagr_rev) else np.nan})
cagr_tbl = pd.DataFrame(cagr_rows).set_index('name')
cagr_tbl = cagr_tbl.sort_values('营收CAGR%', ascending=False)
print('\n[多年复合增速 CAGR · 静态选股依据]')
print(cagr_tbl.round(1).to_string())

# ==== 3. 高成长桶 vs 低成长桶 月度等权回测 ====
# 分组按「事前的行业/业务属性」:5 只公认的成长行业(光伏/锂电/疫苗/制冷部件/模拟芯片) vs 5 只公认的稳健低成长(铁路/水电/港口/高速/贸易)
# 这是事前可知的分类(不是按已实现收益挑,避免未来函数);上面的营收CAGR表已印证两组成长性差异巨大(高16~113% vs 低0~18%)
# 注:单季净利润同比 YOYNI 太吵(高基数股一个季度暴增往往是低基数反弹),不适合当月度选股信号,这里用稳定的行业分桶看成长因子的本色
def bucket_eq(names):
    cols = [n for n in names if n in mret.columns]
    eq, dates = [1.0], [monthly.index[0]]
    for i in range(len(monthly.index) - 1):
        r = mret.iloc[i][cols].mean()
        eq.append(eq[-1] * (1 + (0.0 if pd.isna(r) else r)))
        dates.append(monthly.index[i + 1])
    return pd.Series(eq, index=dates)


eq_hi = bucket_eq(list(HIGH))        # 高成长行业 5 只等权
eq_lo = bucket_eq(list(LOW))         # 低成长行业 5 只等权
eq_bench = bucket_eq(list(STOCKS))   # 10 只等权基准
ret_hi = (eq_hi.iloc[-1] - 1) * 100
ret_lo = (eq_lo.iloc[-1] - 1) * 100
ret_bench = (eq_bench.iloc[-1] - 1) * 100
print('\n[高成长桶 vs 低成长桶 vs 基准 · 2018-2024 月度等权]')
print('高成长桶(光伏/锂电/疫苗/制冷部件/模拟芯片) 总收益 %+.1f%%   终值 %.3f' % (ret_hi, eq_hi.iloc[-1]))
print('低成长桶(铁路/水电/港口/高速/贸易)         总收益 %+.1f%%   终值 %.3f' % (ret_lo, eq_lo.iloc[-1]))
print('基准(10只等权)                            总收益 %+.1f%%   终值 %.3f' % (ret_bench, eq_bench.iloc[-1]))
print('成长溢价 = %+.1f 个百分点(高成长桶 - 低成长桶)' % (ret_hi - ret_lo))

# ==== 4. 估值敏感:高成长桶 vs 低成长桶 的年化波动率 + 最大回撤 + 月度平均 peTTM ====
def _ann_vol(eq):
    m = eq.pct_change().dropna()
    return float(m.std() * np.sqrt(12) * 100)


def _max_dd(eq):
    roll = eq.cummax()
    return float(((eq / roll) - 1).min() * 100)


vol_hi, vol_lo = _ann_vol(eq_hi), _ann_vol(eq_lo)
dd_hi, dd_lo = _max_dd(eq_hi), _max_dd(eq_lo)
pe_hi_ts = pe_m[list(HIGH)].mean(axis=1)            # 高成长桶 月度平均 peTTM
pe_lo_ts = pe_m[list(LOW)].mean(axis=1)             # 低成长桶 月度平均 peTTM
pe_hi_std = float(pe_hi_ts.std())
pe_lo_std = float(pe_lo_ts.std())
print('\n[估值敏感:成长股涨得猛但更颠]')
print('高成长桶 年化波动 %.1f%% / 最大回撤 %.1f%% / 月均PE波动(std) %.1f' % (vol_hi, dd_hi, pe_hi_std))
print('低成长桶 年化波动 %.1f%% / 最大回撤 %.1f%% / 月均PE波动(std) %.1f' % (vol_lo, dd_lo, pe_lo_std))
print('结论:成长桶涨得多但回撤更深、PE 大起大落,估值对情绪/利率更敏感')

# ==== 5. 三张图 ====
# 图1:净值曲线 高成长桶 vs 低成长桶 vs 基准
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(eq_hi.index, eq_hi.values, lw=2, c='#c62828', label='高成长桶(成长行业5只)')
ax.plot(eq_lo.index, eq_lo.values, lw=2, c='#1565c0', label='低成长桶(稳健行业5只)')
ax.plot(eq_bench.index, eq_bench.values, lw=1.5, c='#777', ls='--', label='基准(10只等权)')
ax.axhline(1.0, c='k', lw=.8)
ax.text(eq_hi.index[-1], eq_hi.iloc[-1], ' %+.0f%%' % ret_hi, color='#c62828', fontweight='bold', va='center')
ax.text(eq_lo.index[-1], eq_lo.iloc[-1], ' %+.0f%%' % ret_lo, color='#1565c0', fontweight='bold', va='center')
ax.set_title('高成长桶 vs 低成长桶(2018-2024):成长涨得多(%+.0f%% vs %+.0f%%)但回撤更深 %.0f%% vs %.0f%%' % (
    ret_hi, ret_lo, dd_hi, dd_lo))
ax.set_ylabel('净值(起点=1)')
ax.legend(loc='upper left')
ax.grid(alpha=.3)
fig.tight_layout()
fig.savefig('chart_01.png', dpi=110)
plt.close()

# 图2:估值敏感 —— 两组月度平均 peTTM 时间序列(高成长 PE 大起大落 vs 低成长平稳)
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(pe_hi_ts.index, pe_hi_ts.values, lw=2, c='#c62828', label='高成长组 月均PE(大起大落)')
ax.plot(pe_lo_ts.index, pe_lo_ts.values, lw=2, c='#1565c0', label='低成长组 月均PE(平稳)')
ax.set_title('成长股估值对情绪/利率更敏感:高成长组PE波动 %.0f vs 低成长组 %.0f,涨得猛也颠得狠' % (pe_hi_std, pe_lo_std))
ax.set_ylabel('滚动市盈率 peTTM(组内月均)')
ax.legend(loc='upper left')
ax.grid(alpha=.3)
fig.tight_layout()
fig.savefig('chart_02.png', dpi=110)
plt.close()

# 图3:10 只票营收CAGR 横向柱状,高成长红 / 低成长蓝,显示选股依据
fig, ax = plt.subplots(figsize=(11, 6))
plot_tbl = cagr_tbl.dropna(subset=['营收CAGR%']).sort_values('营收CAGR%')
colors = ['#c62828' if g == '高成长' else '#1565c0' for g in plot_tbl['组别']]
ax.barh(plot_tbl.index, plot_tbl['营收CAGR%'], color=colors)
for i, (idx, v) in enumerate(plot_tbl['营收CAGR%'].items()):
    ax.text(v + (0.5 if v >= 0 else -0.5), i, '%.0f%%' % v, va='center',
            ha='left' if v >= 0 else 'right', fontweight='bold', fontsize=9)
ax.axvline(0, c='k', lw=.8)
ax.set_title('用多年营收复合增速(CAGR)把高成长(红)和低成长(蓝)分开:这就是选股依据')
ax.set_xlabel('营收 CAGR(%,复合年增长率)')
ax.grid(alpha=.3, axis='x')
fig.tight_layout()
fig.savefig('chart_03.png', dpi=110)
plt.close()

print('\n[done] 成长因子高低分组 + 估值敏感实证完成 —— 3 图已出')

## 真实市场案例

| 市场 | 标的 | 实战观察 |
| --- | --- | --- |
| 成长溢价·高成长大胜 | N/A | 2018到2024这7 年,高成长桶(光伏、锂电、疫苗、制冷部件、模拟芯片5只)总收益+282.9%、终值3.829,大幅跑赢低成长桶(铁路、水电、港口、高速、贸易5只)+69.5%、终值1.695,成长溢价+213个点。这10 年成长大胜,恰好是上一节价值跑输的反面。 |
| 估值敏感·涨得多也颠得狠 | N/A | 高成长桶涨得多,但波动和回撤都大得多:年化波动38.5%,差不多是低成长桶18.9%的两倍;最大回撤-59%几乎腰斩,而低成长桶最深也才-33%。高成长桶净值2021年一度冲到约7.4倍,随后崩回3倍多,坐了一趟过山车。 |
| 市盈率大起大落·估值不稳 | N/A | 成长股的市盈率对情绪和利率极敏感。高成长组月均市盈率在大约50到410倍之间疯狂来回甩,波动是85.8;低成长组几乎平趴在10倍附近,波动只有2.7,两者差约30 倍。情绪一来估值吹上天,情绪一走估值就崩,这是成长股最大的风险来源。 |
| 营收增速选股·晶澳vs上港 | sh.600018 | 分组依据是多年营收复合增速:高成长里晶澳科技约113%、智飞生物约53%,几年营收翻好几倍;低成长里上港集团约0%、大秦铁路约4%,业务稳定基本不增长。注意边界处三花17%和建发18%几乎一样,说明成长是连续谱,主要按事前行业属性分桶,避免偷看未来。 |


## 常见坑

### ⚠ 01. 只看某一年的高增长就追,结果买在增速最高点

单年增速会暴涨暴跌,某一年同比涨百分之两百往往是低基数造成的,不可持续;要用多年CAGR平滑着看,别被一年的暴涨骗进去。

### ⚠ 02. 追高成长只盯涨幅,忽略它的波动和回撤

成长股涨得猛也跌得狠,只看历史涨幅不看回撤,容易在情绪退潮、利率抬头时被深度套牢,账户大起大落扛不住。

### ⚠ 03. 用还没披露的财报增速选股,偷看了未来

财报有披露滞后,增速信号必须挂到披露日才算已知。直接用报告期末当天的增速分组,就是前视偏差,回测会虚高。

### ⚠ 04. 亏损公司也硬套复合增长率,算出一堆乱码

利润为负或中途由负转正时,复合增长率的公式会失效甚至算出负数开方的错误;遇到非正的利润必须跳过或改用别的口径。

### ⚠ 05. 把成长当万能,忽略估值已经透支未来

成长公司再好,如果市盈率已经把未来好多年的增长都提前算进去了,买在最贵的位置照样亏钱;成长要和估值、质量一起看。

## 实战 SOP · 成长因子 — 几条铁规矩

1. 成长因子盯赚钱的速度:营收同比、净利润同比看单年,净利/营收复合增速CAGR看长期,优先用多年CAGR平滑暴涨暴跌。
2. 用财报增速选股,信号必须挂到披露日才算已知,绝不用报告期末当天偷看还没公布的财报。
3. 算复合增长率时,起始值和末尾值都必须是正的,亏损或由负转正的公司要单独处理,别硬套公式。
4. 追成长股不能只看涨幅,必须同时看年化波动和最大回撤,涨得快的票往往也是跌得狠的票。
5. 成长要和估值、质量搭着看:增速猛但估值已透支未来、或靠烧钱换增长的,都要警惕。

> 把这段打印贴在你电脑边,执行 1000 次它会回报你。

## 总结 · 你应该带走的

2. 成长因子=用营收、利润增速给票打分,涨得越快分越高,赌好公司会越长越大。
3. 最常用的尺子是营收复合增速:一家公司过去几年营收平均每年涨多少,涨得越快越得分。
4. 诚实结果:2018-2024这7 年,高成长桶+282.9%大幅跑赢低成长桶+69.5%,成长溢价213个点,正好是上节价值跑输的反面。
5. 涨得多也颠得狠:高成长桶年化波动38.5%约是低成长18.9%的两倍,最大回撤-59%几乎腰斩。
6. 成长股估值对情绪利率极敏感:高成长组月均市盈率波动85.8,是低成长组2.7的约30 倍,在50到410倍间大起大落。
7. 方法纪律:单季净利润同比太吵,改用事前行业属性分桶,不看已实现收益、避免未来函数,财报按披露日对齐。

## 自测题

**Q1.** 一家奶茶店去年营收一百万、今年一百三十万,它的营收同比增长是多少?为什么营收同比能反映一家公司在变大还是变小?

**Q2.** 为什么算成长要用多年的复合增长率CAGR,而不是只看某一年的同比?CAGR 比单年增速好在哪里?

**Q3.** 为什么成长股对利率和市场情绪特别敏感、涨得猛也跌得狠?这跟它的高市盈率有什么关系?

把答案写下来,3 天后再回看。

## 下一节预告

**Day 070 · 质量因子** (Quality Factors)

这一节我们用成长因子挑长得快的公司,也看清了成长股涨得猛、颠得狠的脾气。下一节我们换个维度,讲质量因子,看一家公司赚钱的质地好不好,净资产收益率ROE、投入资本回报率ROIC、自由现金流,挑那些不靠烧钱、实打实把钱赚得干净的好公司。

## 推荐阅读

- Eugene Fama, Kenneth French《A Five-Factor Asset Pricing Model》(2015/JFE) — 在三因子基础上加入盈利和投资因子,成长相关因子的现代框架源头。
- Robert Novy-Marx《The Other Side of Value: The Gross Profitability Premium》(2013/JFE) — 把盈利成长拆得很透,说明光看增长不够,还要看赚钱的质地。
- Philip Fisher《Common Stocks and Uncommon Profits》(1958/Harper) — 成长股投资的奠基之作,讲怎么找那些能持续高增长的好公司。
- Peter Lynch《One Up on Wall Street》(1989/Simon & Schuster) — 散户视角找高成长股的经典,用 PEG 把成长和估值放一起看,通俗实用。
- baostock 官方文档 — query_growth_data 净利润同比、query_profit_data 营收和净利润字段说明,动手算成长因子的第一步。